# Agente Explicador de Posições de Brazilian Jiu-Jitsu (BJJ) para faixas brancas com RAG (LangChain)

## Curso: LangChain — Criando Chatbots com RAG (Foundation)

### Objetivo
Construir um agente conversacional simples utilizando **Retrieval-Augmented Generation (RAG)** para responder perguntas sobre **posições de Brazilian Jiu-Jitsu (BJJ) para faixas brancas praticantes**, com base em **documentos especializados**.

---

### Conceitos abordados
- Definição do problema
- Seleção da base de conhecimento
- Preparação dos documentos
- Embeddings e banco vetorial
- Recuperação de contexto (Retriever)
- Integração com LLM
- Testes e validação das respostas



## Dependências

Execute no terminal antes de rodar o notebook:

```bash
pip install langchain langchain-community langchain-openai chromadb pypdf
```


In [21]:

# Importações básicas
import os

# Loader de documentos PDF
from langchain_community.document_loaders import PyPDFLoader

# Divisão de texto em blocos
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embeddings
from langchain_openai import OpenAIEmbeddings

# Banco vetorial
from langchain_community.vectorstores import Chroma

# LLM
from langchain_openai import ChatOpenAI

# Cadeia RAG
from langchain.chains import RetrievalQA



## 1️⃣ Definição do Problema

LLMs possuem conhecimento estático e podem alucinar.
O objetivo aqui é garantir **respostas confiáveis**, conectando o modelo
aos documentos especializados sobre posições básicas de Jiu-Jitsu.

In [26]:

# Caminho do PDF com manual de posições faixa branca de Jiu-Jitsu
CAMINHO_PDF = "manual_posicoes_faixa_branca_jiu_jitsu.pdf"  # ajuste o caminho se necessário  

# Carrega o PDF
loader = PyPDFLoader(CAMINHO_PDF)
documents = loader.load()

# Quantidade de páginas carregadas
len(documents)


8

In [25]:
documents ## O que é ?

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-07-30T01:40:21+00:00', 'author': 'Base de conhecimento RAG', 'keywords': '', 'moddate': '2026-07-30T01:40:21+00:00', 'subject': '(unspecified)', 'title': 'Manual de Posições Recomendadas para Faixa Branca de Jiu-Jitsu', 'trapped': '/False', 'source': 'manual_posicoes_faixa_branca_jiu_jitsu.pdf', 'total_pages': 8, 'page': 0, 'page_label': '1'}, page_content='Manual de Posições Recomendadas para\n Faixa Branca de Jiu-Jitsu\nDocumento de referência técnica organizado para consulta e para uso como base de\nconhecimento (RAG). Conteúdo reorganizado e reescrito a partir de práticas\ncomumente ensinadas a praticantes iniciantes de Jiu-Jitsu, cobrindo rolamentos,\nquedas, passagens de guarda, raspagens, finalizações e posições de controle.\nFonte de referência original: Manual do faixa branca de Jiu Jitsu — Muito Mais Ação Jiu Jitsu (muitomaisacaojiujitsu.com.br).\nEste doc


## 2️⃣ Preparação dos Documentos

Os documentos precisam ser divididos em pequenos blocos
para facilitar a recuperação de contexto.


In [46]:

# Divide os documentos em chunks menores
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=120
)

chunks = text_splitter.split_documents(documents)

len(chunks)


48

In [47]:
chunks ## mostrar 3 pedaços

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-07-30T01:40:21+00:00', 'author': 'Base de conhecimento RAG', 'keywords': '', 'moddate': '2026-07-30T01:40:21+00:00', 'subject': '(unspecified)', 'title': 'Manual de Posições Recomendadas para Faixa Branca de Jiu-Jitsu', 'trapped': '/False', 'source': 'manual_posicoes_faixa_branca_jiu_jitsu.pdf', 'total_pages': 8, 'page': 0, 'page_label': '1'}, page_content='Manual de Posições Recomendadas para\n Faixa Branca de Jiu-Jitsu\nDocumento de referência técnica organizado para consulta e para uso como base de\nconhecimento (RAG). Conteúdo reorganizado e reescrito a partir de práticas\ncomumente ensinadas a praticantes iniciantes de Jiu-Jitsu, cobrindo rolamentos,\nquedas, passagens de guarda, raspagens, finalizações e posições de controle.'),
 Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-07-30T01:40


## 3️⃣ Embeddings e Banco Vetorial

Cada bloco de texto será convertido em vetores semânticos
e armazenado em um banco vetorial.


In [48]:

# Inicializa embeddings
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=os.getenv("OPENAI_API_KEY")
)

# Cria o banco vetorial
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_manual_posicoes_faixa_branca_jiu_jitsu"
)



## 4️⃣ Recuperação de Contexto (Retriever)

O retriever busca os trechos mais relevantes
para cada pergunta do usuário.


In [49]:

# Cria o retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)



## 5️⃣ Integração com o LLM (RAG)

O contexto recuperado será injetado no prompt
enviado ao modelo de linguagem.


In [50]:

# Inicializa o modelo de linguagem
llm = ChatOpenAI(
    model="gpt-4o-mini",
    openai_api_key=os.getenv("OPENAI_API_KEY")
)

# Cria a cadeia RAG
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True
)



## 6️⃣ Testes e Validação

Agora podemos testar perguntas reais
e validar se as respostas estão baseadas nos documentos.


In [52]:
# Pergunta de teste
pergunta = "Qual um dos quatro rolamentos essenciais que um faixa branca deve aprender primeiro?"

# Executa a pergunta no agente RAG
resposta = qa_chain(pergunta)

print("Pergunta:")
print(pergunta)

print("\nResposta do Agente:")
print(resposta["result"])

print("\nTrechos utilizados como contexto:\n")

for i, doc in enumerate(resposta["source_documents"], start=1):
    print(f"--- Trecho {i} ---")
    print(f"Fonte: {doc.metadata.get('source', 'Documento desconhecido')}")
    print(f"Página: {doc.metadata.get('page', 'N/A')}")
    print("Conteúdo:")
    print(doc.page_content)
    print("\n")


Pergunta:
Qual um dos quatro rolamentos essenciais que um faixa branca deve aprender primeiro?

Resposta do Agente:
Um dos quatro rolamentos essenciais que um faixa branca deve aprender primeiro é o rolamento lateral (queda de lado).

Trechos utilizados como contexto:

--- Trecho 1 ---
Fonte: manual_posicoes_faixa_branca_jiu_jitsu.pdf
Página: 1
Conteúdo:
Antes de qualquer técnica de ataque, o faixa branca precisa aprender a cair com segurança. Os
rolamentos são a base para treinar sem se machucar e para reagir bem a quedas e projeções durante o
treino ou a competição. Um aluno que ainda não rola bem tende a se machucar com facilidade nas
quedas.
2.1 Rolamento lateral (queda de lado)
Deitado de costas, com os joelhos formando um ângulo de aproximadamente 90 graus, uma perna


--- Trecho 2 ---
Fonte: manual_posicoes_faixa_branca_jiu_jitsu.pdf
Página: 6
Conteúdo:
professor. Os graus seguintes costumam seguir intervalo semelhante, até completar os 4 graus da
faixa.
Quantos graus tem a faix

In [45]:
# Pergunta de teste 2
pergunta = "Quanto tempo leva, em média, para um faixa branca conquistar o primeiro grau, e quantos graus tem a faixa branca no total?"

# Executa a pergunta no agente RAG
resposta = qa_chain(pergunta)

print("Pergunta:")
print(pergunta)

print("\nResposta do Agente:")
print(resposta["result"])

print("\nTrechos utilizados como contexto:\n")

for i, doc in enumerate(resposta["source_documents"], start=1):
    print(f"--- Trecho {i} ---")
    print(f"Fonte: {doc.metadata.get('source', 'Documento desconhecido')}")
    print(f"Página: {doc.metadata.get('page', 'N/A')}")
    print("Conteúdo:")
    print(doc.page_content)
    print("\n")


Pergunta:
Quanto tempo leva, em média, para um faixa branca conquistar o primeiro grau, e quantos graus tem a faixa branca no total?

Resposta do Agente:
Em média, leva de 4 a 6 meses de treino constante para um faixa branca conquistar o primeiro grau. A faixa branca tem um total de até 4 graus, que são indicados por fitas pretas na ponteira da faixa.

Trechos utilizados como contexto:

--- Trecho 1 ---
Fonte: manual_posicoes_faixa_branca_jiu_jitsu.pdf
Página: 6
Conteúdo:
kimura, armlock, omoplata). São proibidas chaves de perna e pé (com exceção pontual da chave de pé
reta em algumas federações, a partir de faixas mais avançadas), chaves de calcanhar e finalizações do
tipo bate-estaca ou cervicais.
Quanto tempo leva para ganhar o primeiro grau na faixa branca?
Em média, de 4 a 6 meses de treino constante, variando conforme a frequência do aluno e o critério do
professor. Os graus seguintes costumam seguir intervalo semelhante, até completar os 4 graus da
faixa.
Quantos graus tem a fai